<a href="https://colab.research.google.com/github/Om-Ranmode/flyrank-ml-internshipPractice/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Om-Ranmode/flyrank-ml-internshipPractice/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Paper Finding 1: Content Freshness & Decay Detection
Finding: Machine learning classifiers can accurately predict page traffic declines prior to significant position drops.

Label Origin Question: How were declining pages labeled? Is the label based on short-term traffic dips (e.g., seasonal fluctuation over 30 days) or long-term structural decay (e.g., 90+ days), and how are algorithm updates filtered out from true content decay?

Validation Design Question: Was the validation split grouped by client domain or executed time-wise? If training and testing sets shared pages from the same client domain, model performance might reflect domain-level authority rather than generalizable content decay patterns.

Paper Finding 2: Rule-Based vs. Learned Model Precision
Finding: Learned models outperform hand-written heuristic rules by roughly 3x on Precision@50 when ranking pages for refresh priority.

Label Origin Question: What exact business thresholds defined the heuristic rule's target set, and was the baseline rule tuned on the same historical split as the model?

Validation Design Question: Does the metric evaluate out-of-time generalization across entirely unseen domain niches, or are multi-topic sites represented in both splits?

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1 Code: Quick dataset inspection for validation audit
import os, sys, subprocess
import pandas as pd
import numpy as np

# Ensure working directory is set
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Dataset loaded: {df.shape[0]} rows across {df['client_id'].nunique() if 'client_id' in df.columns else df['domain_hash'].nunique()} unique clients.")

Dataset loaded: 30000 rows across 32 unique clients.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Split Comparison: Random Split vs. Grouped Client Split
Random Row-Level Split (Naive): Mixing rows from the same domains across train and test sets leads to data leakage, as the model memorizes domain-level traits, resulting in artificially inflated performance metrics.

Grouped Client Split (Honest): Grouping by client_id ensures that all pages belonging to a specific client domain exist exclusively in either the train or test set, measuring true model generalization to unseen websites.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2 Code: Before/After Comparison (Random Split vs Grouped Split)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Target & Feature Definition (Target Leakage Columns Excluded)
if "target" not in df.columns:
    df["target"] = (df["trend_direction"] == "down").astype(int)

group_col = "client_id" if "client_id" in df.columns else "domain_hash"
leakage_cols = ["target", "id", "trend_direction", "trend_pct"]
features = [c for c in df.select_dtypes(include=[np.number]).columns if c not in leakage_cols]

# -------------------------------------------------------------
# BEFORE: Naive Random Split (Data Leakage Across Domains)
# -------------------------------------------------------------
X_train_rnd, X_test_rnd, y_train_rnd, y_test_rnd = train_test_split(
    df[features], df["target"], test_size=0.2, random_state=42
)

model_rnd = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
model_rnd.fit(X_train_rnd.fillna(0), y_train_rnd)
preds_rnd = model_rnd.predict(X_test_rnd.fillna(0))

# -------------------------------------------------------------
# AFTER: Honest Grouped Split by Client Domain
# -------------------------------------------------------------
np.random.seed(42)
unique_clients = df[group_col].unique()
train_clients = np.random.choice(unique_clients, size=int(0.8 * len(unique_clients)), replace=False)

train_df = df[df[group_col].isin(train_clients)].dropna(subset=features + ["target"])
test_df = df[~df[group_col].isin(train_clients)].dropna(subset=features + ["target"])

X_train_grp, y_train_grp = train_df[features], train_df["target"]
X_test_grp, y_test_grp = test_df[features], test_df["target"]

model_grp = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
model_grp.fit(X_train_grp, y_train_grp)
preds_grp = model_grp.predict(X_test_grp)

# -------------------------------------------------------------
# Results Summary
# -------------------------------------------------------------
split_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Naive Random Split": [
        accuracy_score(y_test_rnd, preds_rnd),
        precision_score(y_test_rnd, preds_rnd, zero_division=0),
        recall_score(y_test_rnd, preds_rnd, zero_division=0),
        f1_score(y_test_rnd, preds_rnd, zero_division=0)
    ],
    "Honest Grouped Split": [
        accuracy_score(y_test_grp, preds_grp),
        precision_score(y_test_grp, preds_grp, zero_division=0),
        recall_score(y_test_grp, preds_grp, zero_division=0),
        f1_score(y_test_grp, preds_grp, zero_division=0)
    ]
})

print("=== Before/After Validation Design Audit ===")
print(split_comparison.round(4).to_string(index=False))

=== Before/After Validation Design Audit ===
   Metric  Naive Random Split  Honest Grouped Split
 Accuracy              0.7525                0.7049
Precision              0.7228                0.6875
   Recall              0.8849                0.9516
 F1-Score              0.7957                0.7983


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Feature Leakage Inspection
Every candidate feature was audited for target contamination:

trend_pct / trend_direction (REMOVED): Directly calculated using post-period traffic changes, representing direct target leakage.

impressions_90d / ctr (KEPT): Historical aggregate metrics available at the time of prediction.

word_count / content_age_days (KEPT): Static metadata independent of future search performance.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 Code: Leakage Audit Check & Feature Correlations with Target
from sklearn.inspection import permutation_importance

# Calculate correlation between remaining features and target
correlations = train_df[features + ["target"]].corr()["target"].drop("target").sort_values(ascending=False)

print("=== Feature Correlation with Target (Post-Audit) ===")
print(correlations.round(4).to_string())

# Permutation importance check on honest model
perm = permutation_importance(model_grp, X_test_grp, y_test_grp, n_repeats=10, random_state=42)
importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": perm.importances_mean
}).sort_values(by="Importance", ascending=False)

print("\n=== Top Feature Importances (Grouped Model) ===")
print(importance_df.head(5).to_string(index=False))

=== Feature Correlation with Target (Post-Audit) ===
days_with_impressions     0.2284
days_since_last_update    0.0754
scroll_rate               0.0444
scroll_events_90d         0.0422
word_count                0.0262
competition               0.0251
avg_position              0.0227
char_count                0.0170
cpc                       0.0082
ai_traffic_pct           -0.0003
ai_sessions_90d          -0.0031
impressions_prev_30d     -0.0111
engagement_rate          -0.0148
search_volume            -0.0198
engaged_sessions_90d     -0.0269
sessions_prev_30d        -0.0309
clicks_prev_30d          -0.0311
pageviews_90d            -0.0317
users_90d                -0.0322
sessions_90d             -0.0328
impressions_90d          -0.0350
days_with_sessions       -0.0378
clicks_90d               -0.0426
ctr                      -0.0570
clicks_last_30d          -0.0822
sessions_last_30d        -0.0886
impressions_last_30d     -0.1207
age_tier_order           -0.1362
content_age_days       

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Audit & Language Refinement
Overly Bold Claim (Initial Draft):

"The Random Forest model proves that search engine traffic declines can be accurately predicted, outperforming standard rules across all websites."

Safe, Public-Ready Rewrite (Decision-Support Language):

"Across the evaluated evaluation dataset, a Random Forest classifier trained on historical impression and position signals demonstrated higher precision in identifying declining pages compared to a fixed heuristic baseline under grouped client validation. These findings represent observed directional signals intended for decision-support workflows rather than deterministic predictions."

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 Code: Error Distribution & Performance Verification
test_df_analysis = test_df.copy()
test_df_analysis["pred"] = preds_grp

fp = len(test_df_analysis[(test_df_analysis["target"] == 0) & (test_df_analysis["pred"] == 1)])
fn = len(test_df_analysis[(test_df_analysis["target"] == 1) & (test_df_analysis["pred"] == 0)])
tp = len(test_df_analysis[(test_df_analysis["target"] == 1) & (test_df_analysis["pred"] == 1)])
tn = len(test_df_analysis[(test_df_analysis["target"] == 0) & (test_df_analysis["pred"] == 0)])

print(f"Verified Error Breakdown on Unseen Clients:")
print(f"True Positives: {tp} | True Negatives: {tn}")
print(f"False Positives (Predicted down, actually stable/up): {fp}")
print(f"False Negatives (Predicted stable/up, actually down): {fn}")

Verified Error Breakdown on Unseen Clients:
True Positives: 3379 | True Negatives: 700
False Positives (Predicted down, actually stable/up): 1536
False Negatives (Predicted stable/up, actually down): 172


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.